# Step 3. 검증 설계 & 함정 2 — 미래를 컨닝하는 랜덤 분할

- **예상 소요 시간:** 약 40분
- **목표:**
  1. 블로그 튜토리얼처럼 `train_test_split`을 **랜덤**으로 썼을 때 점수가 얼마나 **과장**되는지 눈으로 확인한다.
  2. 시간 순서대로 **과거로 학습하고 미래를 검증(시계열 분할)**했을 때 진짜 실력이 어떻게 드러나는지 비교한다.

---

### 백엔드 개발자 관점으로 이해하기
운영 서버에 올릴 API는 **과거(어제까지) 데이터로 학습**해서 **미래(내일) 대여량**을 맞춰야 한다.  
그런데 전체 2년 치 데이터를 랜덤으로 8:2로 섞으면 어떤 일이 벌어질까?

- **2012년 8월 15일 10시** 데이터 → 학습(Train)에 들어감
- **2012년 8월 15일 11시** 데이터 → 시험(Test)에 들어감
- **2012년 8월 15일 12시** 데이터 → 학습(Train)에 들어감

10시(320대)와 12시(350대)를 이미 다 보고 나서 11시 대여량을 맞추는 셈이다.  
실제 운영 서버에서는 12시 데이터가 아직 존재하지 않는다. **미래 데이터를 컨닝한 가짜 점수**가 나온다.

---
## 1. 데이터 로드 & 랜덤 분할의 함정 직접 확인하기

먼저 Step 2와 똑같이 데이터를 불러온 뒤, 똑똑한 모델(`RandomForestRegressor`)을 **랜덤 분할**로 먼저 돌려보자.

> **[실행 전 예측]**  
> 아까 Step 2 선형회귀(랜덤 분할) 오차가 `104.80대`였다.  
> 이번엔 랜덤 분할 그대로 두고 모델만 `RandomForestRegressor`로 바꿔본다. 오차가 몇 대까지 떨어질까?

In [1]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) dataset/hour.csv가 없으면 UCI 공식 저장소에서 자동 다운로드 (로컬 + Colab 공용)
os.makedirs("dataset", exist_ok=True)
if not os.path.exists("dataset/hour.csv"):
    url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
    zip_path = "dataset/bike_sharing_dataset.zip"
    print("데이터 다운로드 중...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("dataset")
    print("다운로드 완료!")

df = pd.read_csv("dataset/hour.csv")
print(f"로드 완료: {df.shape[0]:,}행 × {df.shape[1]}컬럼")

로드 완료: 17,379행 × 17컬럼


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

clean_cols = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed"
]

X = df[clean_cols]
y = df["cnt"]

# [잘못된 검증] 시간 데이터인데 랜덤으로 80:20 섞기
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 1) 선형회귀 (랜덤 분할)
lr_rand = LinearRegression().fit(X_tr_rand, y_tr_rand)
mae_lr_rand = mean_absolute_error(y_te_rand, lr_rand.predict(X_te_rand))

# 2) RandomForest (랜덤 분할)
rf_rand = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_rand.fit(X_tr_rand, y_tr_rand)
mae_rf_rand = mean_absolute_error(y_te_rand, rf_rand.predict(X_te_rand))

print(f"[랜덤 분할 - 선형회귀]      MAE: {mae_lr_rand:.2f}대")
print(f"[랜덤 분할 - RandomForest]  MAE: {mae_rf_rand:.2f}대  <-- 엄청 잘 나오는 것처럼 보임!")

[랜덤 분할 - 선형회귀]      MAE: 104.80대
[랜덤 분할 - RandomForest]  MAE: 24.90대  <-- 엄청 잘 나오는 것처럼 보임!


### 🔍 코드 뜯어보기
- **`RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)`:**  
  - 결정트리(스무고개 나무) **100그루(`n_estimators=100`)**를 만들어 각자 예측하게 한 뒤, 100개 예측값의 평균을 내는 앙상블 모델이다. (자세한 원리는 Step 4에서 다룬다.)
  - `n_jobs=-1`: 내 컴퓨터(또는 Colab)의 **모든 CPU 코어를 병렬로 다 써서** 빠르게 학습하라는 옵션이다. Java의 `parallelStream()` 느낌이다.
- **`.fit(...).predict(...)`:**  
  `LinearRegression`에서 `RandomForestRegressor`로 클래스 이름만 바꿨는데 `.fit()`과 `.predict()` 사용법이 100% 똑같다. `sklearn`의 가장 큰 장점이다.

---
## 2. 진짜로 컨닝하고 있나? 증거 잡기

말로만 "10시와 12시를 보고 11시를 맞춘다"고 하면 와닿지 않는다.  
방금 랜덤 분할된 `X_te_rand`의 첫 번째 테스트 행 바로 앞뒤 1시간이 학습(`X_tr_rand`)에 들어 있는지 직접 조회해 보자.

In [3]:
# 테스트셋 첫 번째 행의 원래 인덱스(행 번호) 꺼내기
sample_idx = X_te_rand.index[0]

# 그 행의 바로 1시간 전(sample_idx - 1), 본인(sample_idx), 1시간 뒤(sample_idx + 1) 확인
check_rows = df.loc[[sample_idx - 1, sample_idx, sample_idx + 1], ["dteday", "hr", "temp", "cnt"]].copy()
check_rows["어디에_들어갔나"] = [
    "Train(학습)" if idx in X_tr_rand.index else "Test(시험)"
    for idx in check_rows.index
]
print(check_rows)

           dteday  hr  temp  cnt   어디에_들어갔나
12829  2012-06-23  18  0.82  481  Train(학습)
12830  2012-06-23  19  0.80  425   Test(시험)
12831  2012-06-23  20  0.78  386  Train(학습)


### 🔍 코드 뜯어보기
- **`X_te_rand.index[0]`:**  
  `train_test_split`으로 쪼개도 각 행의 **원래 행 번호(인덱스)**가 유지된다. `hour.csv`는 시간 순서대로 정렬되어 있으므로 `sample_idx - 1`은 1시간 전, `sample_idx + 1`은 1시간 뒤 데이터다.
- **리스트 컴프리헨션 (`[결과 if 조건 else 결과 for 변수 in 반복대상]`):**  
  파이썬에서 정말 자주 쓰이는 한 줄 `for`문이다. Java로 풀면 아래와 100% 같다.
  ```java
  List<String> labels = new ArrayList<>();
  for (int idx : checkRows.index) {
      labels.add(trainIndex.contains(idx) ? "Train(학습)" : "Test(시험)");
  }
  ```

---
## 3. 제대로 된 검증 — 날짜 기준 시계열 분할 (`2012-09-01` 기점)

이제 실무와 똑같이 **시간 순서대로 칼같이 자른다.**
- **학습(Train):** `2011-01-01` ~ `2012-08-31` (과거 20개월, 약 83.4%)
- **시험(Test):** `2012-09-01` ~ `2012-12-31` (미래 4개월, 약 16.6%)

> **[실행 전 예측]**  
> 똑같은 `RandomForest` 모델이다. 코드 한 줄 안 바꾸고 **분할 방식만 랜덤 → 시계열**로 바꿨다.  
> 아까 랜덤 분할에서 **`24.90대`**였던 오차가 시계열 분할에서는 몇 대로 바뀔까? (26대? 35대? 47대?)

In [4]:
# 날짜 기준으로 과거(Train) / 미래(Test) 칼같이 분리
split_date = "2012-09-01"
train_df = df[df["dteday"] < split_date]
test_df = df[df["dteday"] >= split_date]

print(f"Train 기간: {train_df['dteday'].min()} ~ {train_df['dteday'].max()} ({len(train_df):,}행, {len(train_df)/len(df)*100:.1f}%)")
print(f"Test  기간: {test_df['dteday'].min()} ~ {test_df['dteday'].max()} ({len(test_df):,}행, {len(test_df)/len(df)*100:.1f}%)")

X_train, y_train = train_df[clean_cols], train_df["cnt"]
X_test, y_test = test_df[clean_cols], test_df["cnt"]

# 0) 시계열 기준 Baseline (무조건 학습 평균 찍기)
base_pred = np.full(len(y_test), y_train.mean())
mae_base_time = mean_absolute_error(y_test, base_pred)

# 1) 선형회귀 (시계열 분할)
lr_time = LinearRegression().fit(X_train, y_train)
mae_lr_time = mean_absolute_error(y_test, lr_time.predict(X_test))

# 2) RandomForest (시계열 분할)
rf_time = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
mae_rf_time = mean_absolute_error(y_test, rf_time.predict(X_test))

print("\n========== 랜덤 분할 vs 시계열 분할 비교 ==========")
print(f"[Baseline 평균 찍기]  시계열 MAE: {mae_base_time:.2f}대")
print(f"[선형회귀]            랜덤 MAE: {mae_lr_rand:.2f}대  →  시계열 MAE: {mae_lr_time:.2f}대")
print(f"[RandomForest]       랜덤 MAE: {mae_rf_rand:.2f}대  →  시계열 MAE: {mae_rf_time:.2f}대")

Train 기간: 2011-01-01 ~ 2012-08-31 (14,491행, 83.4%)
Test  기간: 2012-09-01 ~ 2012-12-31 (2,888행, 16.6%)

========== 랜덤 분할 vs 시계열 분할 비교 ==========
[Baseline 평균 찍기]  시계열 MAE: 170.51대
[선형회귀]            랜덤 MAE: 104.80대  →  시계열 MAE: 136.23대
[RandomForest]       랜덤 MAE: 24.90대  →  시계열 MAE: 47.48대


### 🔍 코드 뜯어보기
- **`df[df["dteday"] < "2012-09-01"]`:**  
  - `dteday` 컬럼이 `'YYYY-MM-DD'` 형식 문자열이라 문자열 크기 비교(`<`, `>=`)가 날짜 순서 비교와 정확히 일치한다.
  - SQL의 `SELECT * FROM hour WHERE dteday < '2012-09-01';`과 100% 같다.
  - `pandas`에서 조건 필터링은 항상 `df[조건식]` 형태를 쓴다.
- **왜 시계열 기준 Baseline(`170.51대`)이 랜덤 기준(`140.08대`)보다 클까?**  
  Step 1 질문 4를 떠올려 보자! 2012년 하반기는 서비스가 성장해서 대여량 자체가 2011년보다 훨씬 크다. 과거 평균(약 176대)으로 2012년 가을(평균 250~300대)을 맞추려니 오차가 더 커진 것이다.

### 결과 해석: 랜덤 분할은 오차를 거의 '절반'으로 속였다!
- RandomForest 오차가 **`24.90대` → `47.48대`**로 **거의 2배(+91%) 폭등**했다.
- 만약 랜덤 분할 점수(`24.90대`)만 믿고 팀장님께 *"평균 25대 오차로 정확합니다! 배포하시죠!"*라고 보고했다면, 운영 배포 첫날부터 오차가 2배로 터졌을 것이다.
- **기억할 공식:**  
  - 데이터에 **시간 순서**가 있고 **미래를 예측**하는 문제라면 `train_test_split(shuffle=True)`은 절대 쓰지 않는다.  
  - 반드시 **과거로 학습, 미래로 테스트**한다.

---
### Step 4로 가는 복선
시계열 분할로 제대로 채점해 보니 한 가지가 확 눈에 띈다.
- **선형회귀:** **`136.23대`** (평균 찍기 `170.51대`보다 겨우 34대 나음)
- **RandomForest:** **`47.48대`** (선형회귀보다 **약 3배 정확!**)

왜 똑같은 데이터인데 모델 하나 바꿨다고 오차가 `136대 → 47대`로 확 떨어졌을까? Step 4에서 눈으로 직접 확인한다.

---
## 4. 직접 해보는 자습 미션 (10분)

- **미션 1:** 분할 기준 날짜를 `2012-09-01` 대신 **`2012-07-01`**(여름~겨울 6개월 테스트)로 바꾸면 `RandomForest`의 시계열 MAE는 어떻게 변할까?
- **미션 2:** 시계열 분할 기준에서 테스트 데이터(`2012-09-01` 이후)의 실제 평균 대여량(`y_test.mean()`)과 학습 데이터의 평균 대여량(`y_train.mean()`) 차이를 출력해 보자.

In [ ]:
# [미션 1 & 2] 분할 날짜를 '2012-07-01'로 바꿔보고 평균 차이 & RF MAE 비교하기
# TODO: 아래에 코드를 작성해 보세요.

<details>
<summary>정답 코드 & 문법 해설 보기</summary>

```python
# 미션 1: 2012-07-01 기준 분할
tr_jul = df[df["dteday"] < "2012-07-01"]
te_jul = df[df["dteday"] >= "2012-07-01"]

rf_jul = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_jul.fit(tr_jul[clean_cols], tr_jul["cnt"])
mae_jul = mean_absolute_error(te_jul["cnt"], rf_jul.predict(te_jul[clean_cols]))
print(f"[2012-07-01 분할] RF Test MAE: {mae_jul:.2f}대 (Test 건수: {len(te_jul):,}행)")

# 미션 2: 2012-09-01 분할 기준 Train vs Test 평균 대여량 비교
print(f"Train 평균: {y_train.mean():.1f}대 vs Test 평균: {y_test.mean():.1f}대")
```

- 테스트 기간을 어떻게 자르느냐에 따라 MAE 절대값이 달라진다. 그래서 **실험 비교(Exp 0 ~ Exp 5)를 할 때는 반드시 분할 기준(`2012-09-01`)을 끝까지 고정**해야 공정한 벤치마크가 된다.
</details>

---
## 5. 30분 발표용 정리 (Step 3 분량 = 발표에서 약 4분)

### ① 내 실험 기록표에 적을 숫자
- `Exp 0 (Baseline, 시계열 분할 2012-09-01)`: MAE **170.51대**
- `Exp 2 (함정 2: RandomForest, 랜덤 분할)`: MAE **24.90대** *(미래 컨닝 착시!)*
- `Exp 3 (시계열 분할 교정, 선형회귀)`: MAE **136.23대**
- `Exp 4b 예고 (시계열 분할 교정, RandomForest)`: MAE **47.48대**

### ② 발표 대본 샘플 (구어체)
> "두 번째로 밟은 함정이 제일 실무적인 실수였습니다. 블로그 예제 그대로 `train_test_split`을 써서 랜덤으로 8:2로 나누고 RandomForest를 돌렸더니, 오차가 **24.9대**까지 떨어졌어요.
>
> 근데 곰곰이 생각해 보니 이상했습니다. 랜덤으로 섞으면 8월 15일 10시랑 12시 데이터는 학습에 들어가고 11시만 시험에 들어가거든요. 10시랑 12시 정답을 옆에서 보고 11시를 맞춘 셈입니다. 실제 테스트 행 하나를 찍어봤더니 진짜 앞뒤 1시간이 학습셋에 들어있었습니다.
>
> 그래서 2012년 9월 1일을 기준으로 과거 20개월로 학습하고 미래 4개월을 맞추게 바꿨습니다. 그랬더니 오차가 **24.9대에서 47.5대로 거의 두 배**가 뛰었습니다. 랜덤 분할 점수만 보고 배포했으면 운영에서 큰일 날 뻔한 거죠."

---
**Step 3 완료!**  
다음 **Step 4**에서는 선형회귀(`136대`)가 왜 망했는지 그래프로 확인하고, 결정트리 과적합(`Train 0.01 vs Test 59`)과 RandomForest(`47.5대`)를 비교한 뒤, **모델이 제일 크게 틀린 날 5건**을 직접 뜯어본다.